# Capstone A — Predictive Maintenance: RUL Estimation from Telemetry
#
**Dataset (โปรดอ่านก่อนใช้งาน):**
- *NASA C-MAPSS Turbofan Engine Degradation Simulation Data Set* (subset FD001) — Saxena, A., Goebel, K., Simon, D., Eklund, N. (2008). "Damage propagation modeling for aircraft engine run-to-failure simulation", *Proc. PHM 2008*; data via NASA PCoE Data Set Repository, https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/ (free, attribution required)
- ไฟล์: `datasets/cmapss_turbofan.zip` → `train_FD001.txt` — 100 เครื่อง × วงจรบินจนเกือบพัง, 3 operating settings + 21 sensor channels ต่อรอบ
- **แมปเข้าเคสเมโทรโลยี:** เครื่องยนต์ ≡ เครื่องมือวัดในบริการ · sensor channels ≡ ช่องวัดของเครื่องมือ · **RUL ≡ เวลาที่เหลือก่อนต้องสอบเทียบซ้ำ** — คำถามเดียวกัน: จากพฤติกรรมการวัดที่ผ่านมา อีกกี่รอบก่อนเครื่องหลุดจากเงื่อนไขใช้งาน
#
## เป้าหมายของแทร็กนี้
#
สร้างโมเดล **regression ทำนาย RUL** (remaining useful life) จากช่องวัดรายรอบ วัดผลแบบซื่อสัตย์ต่อเวลา และอธิบายผลได้ว่าช่องวัดไหนบอก "ใกล้พัง" ก่อนเพื่อน

In [ ]:
from pathlib import Path

def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise FileNotFoundError("ไม่พบ datasets/ — รันโน้ตบุ๊กจากใน repo")

DATA = data_dir()

## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)
#
ไฟล์ `.txt` เป็นตัวเลขคั่นช่องว่างไม่มีหัวคอลัมน์ — โครงคือ `unit, cycle, os_1..os_3, s1..s21` (operating settings 3 ค่า + sensor channels 21 ช่อง) และ RUL ต้องสร้างเอง: **RUL ณ รอบหนึ่ง = รอบสุดท้ายของเครื่องนั้น − รอบปัจจุบัน**

In [ ]:
import io
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)
plt.rcParams["font.family"] = ["DejaVu Sans", "Noto Sans Thai", "Leelawadee UI", "Tahoma", "Thonburi"]

COLS = ["unit", "cycle"] + [f"os_{i}" for i in (1, 2, 3)] + [f"s{i}" for i in range(1, 22)]

with zipfile.ZipFile(DATA / "cmapss_turbofan.zip") as z:
    tr = pl.read_csv(io.BytesIO(z.read("train_FD001.txt")), separator=" ", has_header=False,
                     new_columns=COLS, schema_overrides=[pl.Float64] * 26,
                     truncate_ragged_lines=True)

maxc = tr.group_by("unit").agg(pl.col("cycle").max().alias("max_cycle"))
df = tr.join(maxc, on="unit").with_columns(
    (pl.col("max_cycle") - pl.col("cycle")).alias("RUL")
)
print(f"{df.shape[0]} แถว | {df['unit'].max():.0f} เครื่อง | RUL 0–{df['RUL'].max():.0f} รอบ")

## ข้อ 1 — รู้จักเครื่องหนึ่งตัวก่อน (การวัดพังอย่างเป็นระบบหน้าตาเป็นอย่างไร)
#
พลอต 3 ช่องวัดที่ทราบว่าไหลตามการสึก (`s2`, `s3`, `s4`) ของ unit 1 ตามรอบ แล้วตอบ: ช่องไหน "ไหลแบบเร่ง" ช่องไหน "ไหลเชิงเส้น"

In [ ]:
# TODO: filter unit == 1, sort by cycle, พลอต s2/s3/s4 สามเส้น (หรือสามแผง) ตาม cycle
# คำถาม: รูปทรงการไหลต่างกันอย่างไร — และเกี่ยวข้องกับ "เร่ง" ของการพังอย่างไร

## ข้อ 2 — เตรียม feature matrix
#
(ก) ทิ้งช่องที่ **flat** (std ≈ 0 ทุกเครื่อง — ไม่มีข้อมูล: `s1, s5, s10, s16, s18, s19`) (ข) แนะนำต่อ: สร้าง rolling features รายเครื่อง (เช่น slope ของ 10 รอบล่าสุดของแต่ละช่อง) ด้วย `group_by("unit")` — เริ่มจากแบบง่าย: ใช้ค่าดิบรายรอบก่อน ถ้ามีเวลาค่อยเพิ่ม rolling

In [ ]:
# TODO: FLAT = ["s1","s5","s10","s16","s18","s19"]
# feature_cols = ทุกคอลัมน์ที่ไม่ใช่ unit/cycle/max_cycle/RUL/os_* และไม่อยู่ใน FLAT
# X = df[feature_cols] → numpy, y = df["RUL"] → numpy

## ข้อ 3 — แบ่งข้อมูลแบบซื่อสัตย์: แบ่ง "เครื่อง" ไม่ใช่ "แถว"
#
แถวของเครื่องเดียวกันเกี่ยวเนื่องกัน — ถ้าสุ่มแบ่ง แถวของเครื่องเดียวหลุดไปทั้ง train และ test (leakage) ให้แบ่งด้วย **unit**: 50 เครื่องแรก train, 50 หลัง test

In [ ]:
# TODO: units 1–50 → train, 51–100 → test (สร้าง mask จาก df["unit"])
# จำแนก: เหตุผลที่ต้องแบ่งระดับ unit (ไม่ใช่แถว) ใน 1 ประโยค

## ข้อ 4 — เทรน + วัดผล
#
เทรน XGBoost regressor ทำนาย RUL วัดด้วย MAE + RMSE บน test เทียบกับ baseline "ทำนาย RUL เฉลี่ยของ train ทุกชิ้น" — **แนะนำเพิ่ม:** พลอต predicted vs true RUL แล้วสังเกตว่าโมเดลอ่อนแค่ไหนเมื่อ RUL จริงสูงมาก (ข้อมูล train ไม่เคยเห็น "RUL สูงสุด" ที่มีความหมาย — เครื่องถูกถอนก่อนพังจริง)

In [ ]:
# TODO: XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
# fit บน train → MAE/RMSE บน test → เทียบ baseline (predict ค่าเฉลี่ย RUL ของ train)
# พลอต predicted vs true (scatter, เส้น y=x) แล้วเขียน 2–3 บรรทัดตีความ

## ข้อ 5 — อธิบายโมเดล (SHAP)
#
คำนวณ SHAP บน test ชุดย่อย (~300 แถว) พลอต beeswarm แล้วตอบ: 3 ช่องวัดที่บอก "ใกล้พัง" ก่อนเพื่อนคืออะไร — และถ้าจะติดตั้ง monitoring จริงบนเครื่องมือของห้องแล็บ จะเลือกเก็บช่องไหนก่อน (เพราะเก็บช่องเดียวไม่ไหว)

In [ ]:
# TODO: shap.TreeExplainer(model)(pd.DataFrame(Zte, columns=feature_cols))
# beeswarm(max_display=10) + ประโยคตอบแบบ Module 9

## ข้อ 6 — สรุปส่งงาน
#
ย่อหน้า 3–5 บรรทัด: โมเดลทำอะไรได้/ไม่ได้, ข้อจำกัดสำคัญ (เช่น train ไม่มีเครื่องที่พังจริง — RUL สูง ๆ คือการ extrapolate), และถ้ามีเวลาอีกสัปดาห์จะทำอะไรต่อ (rolling features? RUL clipping ที่ ~125–130 รอบตามวรรณกรรม? โมเดลต่อเครื่อง?)